In [ ]:
# --- Cell 1: environment + scratch space discovery -------------------------
import os, sys, platform, subprocess

print("project:", "Underdog-Saluki-27B")
print("python:", sys.version.split()[0])
print("platform:", platform.platform())

for p in ("/kaggle/temp", "/kaggle/working"):
    os.makedirs(p, exist_ok=True)
    st = os.statvfs(p)
    free = st.f_bavail * st.f_frsize / 1e9
    print(f"{p}: free={free:.1f} GB writable={os.access(p, os.W_OK)}")

print("--- GPU ---")
print(subprocess.run(["nvidia-smi",
                      "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout or "no nvidia-smi")

print("--- internet ---")
for u in ("https://huggingface.co", "https://github.com"):
    r = subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}",
                        "--max-time", "30", u], capture_output=True, text=True)
    print(u, "->", r.stdout)


In [ ]:
# --- Cell 2: download the GGUF (ON KAGGLE) ---------------------------------
# Optional companions (vision projector, speculative drafter) are pulled too.
import os, subprocess, sys

MODEL_REPO = "ConwayResearch/Underdog-Saluki-27B-1.0"
MODEL_FILE = "Underdog-Saluki-27B-1.0-IQ2-mix.gguf"
MODEL_SIZE_GB = float("7.89")
MMPROJ_FILE = ""
MMPROJ_SIZE_GB = float("" or "0")
DRAFT_FILE = ""
DRAFT_SIZE_GB = float("" or "0")

MODEL_DIR = "/kaggle/temp/models"
os.makedirs(MODEL_DIR, exist_ok=True)


def curl_download(repo, filename, dest_dir):
    """Resume-capable curl download from the HF resolve endpoint."""
    out = os.path.join(dest_dir, os.path.basename(filename))
    url = f"https://huggingface.co/{repo}/resolve/main/{filename}"
    print(f"downloading {filename} -> {out}", flush=True)
    subprocess.run(["curl", "-L", "-C", "-", "--retry", "10", "--retry-delay", "5",
                    "--progress-bar", "-o", out, url], check=True)
    return out


def hf_download(repo, filename, dest_dir):
    """Preferred path: huggingface_hub (+ hf_transfer when installable)."""
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hf_transfer"],
                       check=False)
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
        from huggingface_hub import hf_hub_download
        return hf_hub_download(repo_id=repo, filename=filename, local_dir=dest_dir)
    except Exception as e:
        print("huggingface_hub path failed -> falling back to curl:", e, flush=True)
        return curl_download(repo, filename, dest_dir)


def ensure(repo, filename, size_gb, label):
    """Download `filename` unless a complete copy is already present."""
    if not filename:
        return ""
    path = os.path.join(MODEL_DIR, os.path.basename(filename))
    floor = int(size_gb * 1e9 * 0.98) if size_gb else 0
    if os.path.exists(path) and (not floor or os.path.getsize(path) >= floor):
        print(f"{label}: cached {path}", flush=True)
        return path
    path = hf_download(repo, filename, MODEL_DIR)
    actual = os.path.getsize(path) / 1e9
    print(f"{label}: {path} ({actual:.2f} GB)", flush=True)
    if floor and os.path.getsize(path) < floor:
        raise RuntimeError(f"{label} looks truncated: {actual:.2f} GB expected ~{size_gb} GB")
    return path


model_path = ensure(MODEL_REPO, MODEL_FILE, MODEL_SIZE_GB, "model")
mmproj_path = ensure(MODEL_REPO, MMPROJ_FILE, MMPROJ_SIZE_GB, "mmproj")
draft_path = ensure(MODEL_REPO, DRAFT_FILE, DRAFT_SIZE_GB, "draft")


In [ ]:
# --- Cell 3: install a prebuilt llama.cpp ----------------------------------
import os, subprocess, glob

LLAMA_BUILD = "b11636"
LLAMA_DIR = "/kaggle/temp/llama"
BASE = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}"
os.makedirs(LLAMA_DIR, exist_ok=True)


def llama_server_path():
    hits = glob.glob(os.path.join(LLAMA_DIR, "**", "llama-server"), recursive=True)
    return hits[0] if hits else None


if not llama_server_path():
    # CUDA build first; the cudart archive carries the matching CUDA runtime libs.
    for asset in (f"llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz",
                  f"cudart-llama-{LLAMA_BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz"):
        subprocess.run(["curl", "-sL", "-o", f"{LLAMA_DIR}/{asset}", f"{BASE}/{asset}"],
                       check=True)
        subprocess.run(["tar", "-xzf", f"{LLAMA_DIR}/{asset}", "-C", LLAMA_DIR], check=True)

LLAMA_SERVER = llama_server_path()
if not LLAMA_SERVER:
    raise RuntimeError("llama-server not found after extraction")

os.chmod(LLAMA_SERVER, 0o755)
LLAMA_BIN_DIR = os.path.dirname(LLAMA_SERVER)
os.environ["LD_LIBRARY_PATH"] = LLAMA_BIN_DIR + ":" + os.environ.get("LD_LIBRARY_PATH", "")

print("llama-server:", LLAMA_SERVER)
print(subprocess.run([LLAMA_SERVER, "--version"], capture_output=True, text=True).stdout)


In [ ]:
# --- Cell 4: start llama-server + publish public tunnels -------------------
import os, re, subprocess, time, urllib.request, json, threading, http.server

PORT = int("8080")
CTL_PORT = int("8081")
CTX = int("32768")
NGL = int("99")
SPLIT_MODE = "layer"
EXTRA_ARGS = "".split()

# Set by the control server's /stop route; cell 5 watches it and exits cleanly.
STOP_REQUESTED = threading.Event()

WORK = "/kaggle/temp/run"
os.makedirs(WORK, exist_ok=True)
LLM_LOG = os.path.join(WORK, "llama-server.log")
CF_LOG = os.path.join(WORK, "cloudflared.log")
CTL_LOG = os.path.join(WORK, "cloudflared-ctl.log")

# ---- cloudflared (public URL) ----
CF = "/kaggle/temp/cloudflared"
if not os.path.exists(CF):
    subprocess.run(["curl", "-sL", "-o", CF,
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    os.chmod(CF, 0o755)

AUTH_USER = "alamgir-zk" or os.environ.get("CLUSTER_AUTH_USER", "")
AUTH_PASS = "Zaid2002@kaggle" or os.environ.get("CLUSTER_AUTH_PASS", "")
AUTH_KEY = "sk-cluster-ai883742-secret-key" or os.environ.get("CLUSTER_API_KEY", "")
AUTH_ENABLED = bool((AUTH_USER and AUTH_PASS) or AUTH_KEY)
LLAMA_TARGET_PORT = PORT if not AUTH_ENABLED else (PORT + 2)

# ---- llama-server (OpenAI-compatible) ----
cmd = [LLAMA_SERVER, "-m", model_path, "--jinja", "-ngl", str(NGL),
       "-fa", "on", "-c", str(CTX), "-np", "1",
       "--host", "127.0.0.1", "--port", str(LLAMA_TARGET_PORT)]
# -np 1 keeps the whole context for a single request (llama-server otherwise
# divides -c across parallel slots, silently shrinking what one generation sees).
if SPLIT_MODE:
    cmd += ["--split-mode", SPLIT_MODE]
if mmproj_path:
    cmd += ["--mmproj", mmproj_path]
if draft_path:
    cmd += ["--model-draft", draft_path, "--gpu-layers-draft", str(NGL)]
cmd += EXTRA_ARGS
print("launching:", " ".join(cmd), flush=True)

llm_log_fh = open(LLM_LOG, "w")
server = subprocess.Popen(cmd, stdout=llm_log_fh, stderr=subprocess.STDOUT,
                          env=os.environ.copy(), cwd=LLAMA_BIN_DIR)

# ---- wait for readiness ----
ready = False
for i in range(300):                      # up to 25 minutes
    time.sleep(5)
    if server.poll() is not None:
        print("llama-server exited early; log tail:", flush=True)
        print(open(LLM_LOG).read()[-4000:], flush=True)
        raise RuntimeError("llama-server died on startup")
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{LLAMA_TARGET_PORT}/health", timeout=4) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        pass
    if i % 6 == 0:
        tail = open(LLM_LOG).read().strip().splitlines()[-1:] or [""]
        print(f"[{i*5}s] loading... {tail[0][:120]}", flush=True)

if not ready:
    print(open(LLM_LOG).read()[-4000:], flush=True)
    raise RuntimeError("llama-server did not become healthy in time")

print("llama-server is UP on port", LLAMA_TARGET_PORT, flush=True)

# ---- authenticating reverse proxy (if auth enabled) ----
if AUTH_ENABLED:
    import base64
    class AuthProxy(http.server.BaseHTTPRequestHandler):
        def log_message(self, *a): pass

        def _check_auth(self):
            path = self.path.split("?")[0].rstrip("/")
            if path in ("/health", "/healthz"):
                return True
            auth = self.headers.get("Authorization", "")
            if auth.startswith("Basic "):
                try:
                    creds = base64.b64decode(auth[6:].strip()).decode("utf-8")
                    u, p = creds.split(":", 1)
                    if u == AUTH_USER and p == AUTH_PASS:
                        return True
                except Exception:
                    pass
            elif auth.startswith("Bearer "):
                token = auth[7:].strip()
                if (AUTH_KEY and token == AUTH_KEY) or (AUTH_PASS and token == AUTH_PASS):
                    return True
            self.send_response(401)
            self.send_header("WWW-Authenticate", 'Basic realm="AI Cluster Protected"')
            self.send_header("Content-Type", "application/json")
            self.send_header("Access-Control-Allow-Origin", "*")
            self.end_headers()
            self.wfile.write(b'{"error": "Unauthorized: valid username and password required"}\n')
            return False

        def do_OPTIONS(self):
            self.send_response(200)
            self.send_header("Access-Control-Allow-Origin", "*")
            self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS, PUT, DELETE")
            self.send_header("Access-Control-Allow-Headers", "Content-Type, Authorization")
            self.end_headers()

        def do_HEAD(self):
            if self._check_auth(): self._forward("HEAD")
        def do_GET(self):
            if self._check_auth(): self._forward("GET")
        def do_POST(self):
            if self._check_auth(): self._forward("POST")

        def _forward(self, method):
            content_len = int(self.headers.get("Content-Length", 0))
            body = self.rfile.read(content_len) if content_len > 0 else None
            headers = {k: v for k, v in self.headers.items() if k.lower() not in ("host", "authorization")}
            req = urllib.request.Request(f"http://127.0.0.1:{LLAMA_TARGET_PORT}{self.path}",
                                         data=body, headers=headers, method=method)
            try:
                with urllib.request.urlopen(req, timeout=600) as r:
                    self.send_response(r.status)
                    for k, v in r.getheaders():
                        if k.lower() not in ("transfer-encoding", "content-length"):
                            self.send_header(k, v)
                    self.end_headers()
                    while True:
                        chunk = r.read(4096)
                        if not chunk: break
                        self.wfile.write(chunk)
                        self.wfile.flush()
            except urllib.error.HTTPError as e:
                self.send_response(e.code)
                for k, v in e.headers.items():
                    if k.lower() not in ("transfer-encoding", "content-length"):
                        self.send_header(k, v)
                self.end_headers()
                self.wfile.write(e.read())

    proxy = http.server.ThreadingHTTPServer(("127.0.0.1", PORT), AuthProxy)
    threading.Thread(target=proxy.serve_forever, daemon=True).start()
    print("Auth Proxy listening on port", PORT, flush=True)

# ---- local control server: lets your PC end the session on demand --------
# Kaggle's public API has no session-stop permission, so this tiny server is
# what `ctr.sh stop` calls. It only listens on 127.0.0.1 and is exposed
# through its own tunnel; the path is secret-ish (random token).
STOP_TOKEN = os.environ.get("CLUSTER_STOP_TOKEN") or os.urandom(8).hex()

class CtlHandler(http.server.BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    def _send(self, code, payload):
        body = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path.rstrip("/") in ("", "/health"):
            self._send(200, {"ok": True, "status": "up",
                             "stopping": STOP_REQUESTED.is_set(),
                             "model": "Underdog-Saluki-27B-1.0-IQ2-mix.gguf"})
        elif self.path.startswith("/stop/" + STOP_TOKEN):
            STOP_REQUESTED.set()
            self._send(200, {"ok": True, "message": "shutdown requested"})
            print("STOP_REQUESTED received via control tunnel", flush=True)
        else:
            self._send(404, {"ok": False, "error": "not found"})

    def do_POST(self):
        self.do_GET()

ctl = http.server.ThreadingHTTPServer(("127.0.0.1", CTL_PORT), CtlHandler)
threading.Thread(target=ctl.serve_forever, daemon=True).start()

# ---- publish both tunnels ----
CF_TOKEN = "eyJhIjoiYzBiNGZhMjIzYWRkOTNhZDAxOTk4YmM0MDhmNzhjZDMiLCJzIjoiNk42Sm1KZERGdUFrUDVHcUpjekdvcnVHbHV0MFdVL0ZPdDFmN3k0SW4rVT0iLCJ0IjoiYmM2ZjA4OGQtYWU4ZC00ZmU3LWEzMmQtMjdjYTA3YWFiZTkwIn0=" or os.environ.get("CLOUDFLARE_TUNNEL_TOKEN", "")
STATIC_URL = "https://ai.883742.xyz" or os.environ.get("CLUSTER_URL", "")

def start_public_tunnel(url, log_path):
    fh = open(log_path, "w")
    if CF_TOKEN:
        print("Starting named Cloudflare tunnel with token...", flush=True)
        proc = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "run", "--token", CF_TOKEN, "--url", url],
                                stdout=fh, stderr=subprocess.STDOUT)
        time.sleep(5)
        if proc.poll() is not None:
            fh.flush()
            print("cloudflared failed to start! Log tail:", flush=True)
            print(open(log_path).read()[-2000:], flush=True)
            raise RuntimeError("named cloudflared tunnel exited immediately")
        return STATIC_URL or "https://ai.883742.xyz"
    else:
        subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", url],
                         stdout=fh, stderr=subprocess.STDOUT)
        for _ in range(60):
            time.sleep(2)
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
            if m:
                return m.group(0)
        return None

def start_tunnel(url, log_path):
    fh = open(log_path, "w")
    subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", url],
                     stdout=fh, stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(2)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(log_path).read())
        if m:
            return m.group(0)
    return None

public_url = start_public_tunnel(f"http://127.0.0.1:{PORT}", CF_LOG)
control_url = start_tunnel(f"http://127.0.0.1:{CTL_PORT}", CTL_LOG)

if not public_url:
    print("cloudflared log:", open(CF_LOG).read()[-2000:], flush=True)
    raise RuntimeError("could not obtain a public tunnel URL")
if not control_url:
    print("control cloudflared log:", open(CTL_LOG).read()[-2000:], flush=True)
    raise RuntimeError("could not obtain a control tunnel URL")

with open("/kaggle/working/CONTROL_URL.txt", "w") as f:
    f.write(public_url + "\n")
with open("/kaggle/working/STOP_URL.txt", "w") as f:
    f.write(control_url + "/stop/" + STOP_TOKEN + "\n")

print("=" * 72)
print("CONTROL_URL=" + public_url)
print("STOP_URL=" + control_url + "/stop/" + STOP_TOKEN)
print("MODEL=" + "Underdog-Saluki-27B-1.0-IQ2-mix.gguf")
print("=" * 72)
print(f"Chat endpoint : {public_url}/v1/chat/completions")
print(f"Health        : {public_url}/health")
print("Model name    : saluki  (any name works; llama-server ignores it)")
print("Stop from PC  : ctr.sh stop   (uses the control tunnel above)")


In [ ]:
# --- Cell 5: keep the tunnel alive (this is what you keep running) ---------
import time, urllib.request

PORT = int("8080")
SERVE_MINUTES = int("660")
AUTH_USER = "alamgir-zk"
AUTH_PASS = "Zaid2002@kaggle"

url = open("/kaggle/working/CONTROL_URL.txt").read().strip()
print(f"serving for ~{SERVE_MINUTES} minutes at {url}", flush=True)

for minute in range(SERVE_MINUTES):
    # Wait up to 60s, but poll the stop flag every 5s so shutdown is prompt.
    for _ in range(12):
        if STOP_REQUESTED.is_set():
            break
        time.sleep(5)
    if STOP_REQUESTED.is_set():
        print("STOP received from control tunnel -> ending session", flush=True)
        break
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/health")
        if AUTH_USER and AUTH_PASS:
            import base64
            auth_str = base64.b64encode(f"{AUTH_USER}:{AUTH_PASS}".encode()).decode()
            req.add_header("Authorization", f"Basic {auth_str}")
        with urllib.request.urlopen(req, timeout=5) as r:
            status = str(r.status)
    except Exception as e:
        status = f"err {e}"
    if minute % 5 == 0:
        # Re-print periodically so the URL can always be recovered from logs.
        print(f"[{minute}m] health={status} CONTROL_URL={url}", flush=True)

print("SERVE_WINDOW_COMPLETE", flush=True)
